In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix
import joblib
import os
import sys

sys.path.append('..')
os.chdir('..')

from src.utils.config_loader import load_config
from src.data_pipeline.preprocess import *
from src.data_pipeline.features import *
from src.models.cf_model import *

config = load_config("configs/data_config.yaml")
print("✅ Imports done")

https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Electronics.jsonl.gz
✅ Imports done


d:\Ahmed\study\DEPI\tasks\Final_project\recommendation-system\system_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
processed_path = config['paths']['processed_data']

# 2. Load Processed DataFrames
print("[INFO] Loading DataFrames...")
train_df = pd.read_parquet(processed_path + 'train.parquet')
val_df   = pd.read_parquet(processed_path + 'val.parquet')

# 3. Load Encoders and Scaler
print("[INFO] Loading Encoders and Scaler...")
encoders = joblib.load(processed_path + 'encoders.pkl')
scaler   = joblib.load(processed_path + 'scaler.pkl')

# 4. Load the Fixed-Shape Sparse Matrices
print("[INFO] Loading Sparse Matrices...")
train_matrix = joblib.load(processed_path + 'train_matrix.pkl')
val_matrix   = joblib.load(processed_path + 'val_matrix.pkl')

print("\n=== Data Overview ===")
print(f"Train DF shape:      {train_df.shape}")
print(f"Validation DF shape: {val_df.shape}")
print(f"Train Matrix shape:  {train_matrix.shape}")

# Preview the first few rows
train_df.head()

[INFO] Loading DataFrames...
[INFO] Loading Encoders and Scaler...
[INFO] Loading Sparse Matrices...

=== Data Overview ===
Train DF shape:      (608593, 14)
Validation DF shape: (98155, 14)
Train Matrix shape:  (177495, 274155)


,rating,title,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,weight,user_segment,user_verified_ratio,item_avg_rating,is_weekend
0,5.0,Five Stars,None,80588,B00L87YMGM,0,2014-09-08 17:50:45,0.0,1,1.0,Medium,1.0,4.384615,0
1,4.0,Four Stars,None,33834,B07PR21291,0,2014-09-08 17:50:59,0.0,1,1.0,Medium,1.0,4.000000,0
2,4.0,Four Stars,Works well with my Pi,77220,B00JO80LUI,0,2014-09-08 17:51:14,0.0,1,1.0,Medium,1.0,3.312500,0
3,5.0,Love these! Will be ordering more,Love these! Will be ordering more.,1663,B001FB6SI6,0,2015-12-14 19:20:29,0.0,1,1.0,Medium,1.0,5.000000,0
4,5.0,Five Stars,None,42776,B0C2HWSXNL,0,2015-12-14 19:24:35,0.0,1,1.0,Medium,1.0,4.701299,0


In [3]:
# Extract the item encoder to decode product IDs
item_encoder = encoders['asin']

# 1. Setup Scenario 1: The First Item (Sparse / Rare)
test_item_idx = 0
test_asin = item_encoder.inverse_transform([test_item_idx])[0]

# 2. Setup Scenario 2: The Most Popular Item (Dense / Frequent)
# Summing interactions across all users for each item
item_interaction_counts = np.asarray(train_matrix.sum(axis=0)).flatten()
popular_item_idx = item_interaction_counts.argmax()
popular_asin = item_encoder.inverse_transform([popular_item_idx])[0]

print(f"[INFO] First Item ASIN: {test_asin} (Interactions: {item_interaction_counts[test_item_idx]})")
print(f"[INFO] Popular Item ASIN: {popular_asin} (Interactions: {item_interaction_counts[popular_item_idx]})")

# 3. Helper Function for Comparison
def print_model_comparison(model, item_idx, asin, model_name):
    recs = model.recommend(item_id=item_idx, n_recommendations=5)
    print(f"\n--- {model_name} Recommendations for {asin} ---")
    for rank, (idx, score) in enumerate(recs, start=1):
        rec_asin = item_encoder.inverse_transform([idx])[0]
        # Adjust score calculation for KNN (distance to similarity)
        if model_name == "Item-Based KNN":
            score = 1 - score 
        print(f"Rank {rank}: ASIN {rec_asin} | Score: {score:.4f}")

[INFO] First Item ASIN: 0060897082 (Interactions: 5.0)
[INFO] Popular Item ASIN: B01G8JO5F2 (Interactions: 17793.0)


In [ ]:
# 1. KNN Model
print("\n[INFO] 1. Training Item-Based KNN.")
knn_model = ItemBasedKNN(n_neighbors=10, metric='cosine')
knn_model.fit(train_matrix)

# 2. ALS Model
print("\n[INFO] 2. Training ALS Model.")
als_model = ALSRecommender(factors=64, iterations=20)
als_model.fit(train_matrix)

# 3. BPR Model
print("\n[INFO] 3. Training BPR Model.")
bpr_model = BPRRecommender(factors=64, iterations=100)
bpr_model.fit(train_matrix)

# 4. SVD Model
print("\n[INFO] 4. Training SVD Model.")
svd_model = SVDRecommender(n_components=64)
svd_model.fit(train_matrix)

print("\n[INFO] All 4 Models successfully trained and loaded in memory.")


[INFO] 1. Training Item-Based KNN...
[INFO] Fitting Item-Based KNN with metric='cosine'...
[INFO] KNN Model fitting complete. ✅

[INFO] 2. Training ALS Model...
[INFO] Fitting ALS Model with 64 latent factors...


d:\Ahmed\study\DEPI\tasks\Final_project\recommendation-system\system_env\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 20/20 [00:12<00:00,  1.63it/s]


[INFO] ALS Model fitting complete. 

[INFO] 3. Training BPR Model...
[INFO] Fitting BPR Model with 64 latent factors...


100%|██████████| 100/100 [00:05<00:00, 18.41it/s, train_auc=77.04%, skipped=0.14%]


[INFO] BPR Model fitting complete. ✅

[INFO] 4. Training SVD Model...
[INFO] Fitting SVD Model with 64 components...
[INFO] SVD Model fitting complete. ✅

[INFO] All 4 Models successfully trained and loaded in memory! ✅


In [5]:

print("="*80)
print(f"========== SCENARIO 1: First Item (ASIN: {test_asin}) ==========")
print("="*80)

print_model_comparison(knn_model, test_item_idx, test_asin, "Item-Based KNN")
print_model_comparison(als_model, test_item_idx, test_asin, "ALS")
print_model_comparison(bpr_model, test_item_idx, test_asin, "BPR")
print_model_comparison(svd_model, test_item_idx, test_asin, "SVD")

========== SCENARIO 1: First Item (ASIN: 0060897082) ==========

--- Item-Based KNN Recommendations for 0060897082 ---
Rank 1: ASIN B08K8STN3Y | Score: 1.0000
Rank 2: ASIN B08DT7JX29 | Score: 1.0000
Rank 3: ASIN B084MR26T3 | Score: 1.0000
Rank 4: ASIN B084MQSQ1P | Score: 1.0000
Rank 5: ASIN B07XCRXFWL | Score: 1.0000

--- ALS Recommendations for 0060897082 ---
Rank 1: ASIN B08K8STN3Y | Score: 1.0000
Rank 2: ASIN B08239TPNS | Score: 1.0000
Rank 3: ASIN B07M64R4CR | Score: 1.0000
Rank 4: ASIN B07G2V1WJ3 | Score: 1.0000
Rank 5: ASIN B072KCS54X | Score: 1.0000

--- BPR Recommendations for 0060897082 ---
Rank 1: ASIN B00ITORJRQ | Score: 0.9861
Rank 2: ASIN B00V7ZY7XI | Score: 0.9835
Rank 3: ASIN B07KSBK92C | Score: 0.9828
Rank 4: ASIN B07ZJRT396 | Score: 0.9801
Rank 5: ASIN B08DT7JX29 | Score: 0.9791

--- SVD Recommendations for 0060897082 ---
Rank 1: ASIN B084MQSQ1P | Score: 1.0000
Rank 2: ASIN B003DKL55S | Score: 1.0000
Rank 3: ASIN B07VXQ4WJ4 | Score: 1.0000
Rank 4: ASIN B019OSCLH8 | Sco

In [6]:
print("="*80)
print(f"========== SCENARIO 2: Most Popular Item (ASIN: {popular_asin}) ==========")
print("="*80)

print_model_comparison(knn_model, popular_item_idx, popular_asin, "Item-Based KNN")
print_model_comparison(als_model, popular_item_idx, popular_asin, "ALS")
print_model_comparison(bpr_model, popular_item_idx, popular_asin, "BPR")
print_model_comparison(svd_model, popular_item_idx, popular_asin, "SVD")

========== SCENARIO 2: Most Popular Item (ASIN: B01G8JO5F2) ==========

--- Item-Based KNN Recommendations for B01G8JO5F2 ---
Rank 1: ASIN B0792QJQT1 | Score: 0.0595
Rank 2: ASIN B07PBS21V8 | Score: 0.0478
Rank 3: ASIN B07D3NS9TQ | Score: 0.0382
Rank 4: ASIN B07KR62YBD | Score: 0.0353
Rank 5: ASIN B083H4Y757 | Score: 0.0321

--- ALS Recommendations for B01G8JO5F2 ---
Rank 1: ASIN B00SX03Q3M | Score: 0.9965
Rank 2: ASIN B083JKDNRJ | Score: 0.9965
Rank 3: ASIN B0784MZBYY | Score: 0.9962
Rank 4: ASIN B07K8WZ19K | Score: 0.9962
Rank 5: ASIN B01D4G9D40 | Score: 0.9962

--- BPR Recommendations for B01G8JO5F2 ---
Rank 1: ASIN B07D3NS9TQ | Score: 0.9779
Rank 2: ASIN B009ES6LU2 | Score: 0.9737
Rank 3: ASIN B07F2DZSP2 | Score: 0.9723
Rank 4: ASIN B07NXPWCG2 | Score: 0.9678
Rank 5: ASIN B00DMGSMY0 | Score: 0.9661

--- SVD Recommendations for B01G8JO5F2 ---
Rank 1: ASIN B00PMLTOXQ | Score: 0.9947
Rank 2: ASIN B07G23R6NN | Score: 0.9947
Rank 3: ASIN B0196BGATI | Score: 0.9947
Rank 4: ASIN B01A6TYUV

In [ ]:
# Define the save path
model_save_path = config['paths']['processed_data']  # Or create a config['paths']['models']

print("[INFO] Saving Models to disk.")

joblib.dump(knn_model, os.path.join(model_save_path, 'knn_baseline.pkl'))
joblib.dump(als_model, os.path.join(model_save_path, 'als_model.pkl'))
joblib.dump(bpr_model, os.path.join(model_save_path, 'bpr_model.pkl'))
joblib.dump(svd_model, os.path.join(model_save_path, 'svd_model.pkl'))

print("[INFO] All models successfully saved")

[INFO] Saving Models to disk...
[INFO] All models successfully saved! 💾✅
